# V8.1 Full Dataset Training - Multilingual Laughter Detection

## Dataset
- **V8.1 Final**: 12,048 examples
  - English: 5,949 train
  - Chinese (zh): 2,051 train  
  - Hindi (hi): 1,600 train
- **Plus StandUp4AI**: 3,203 words (fr, en, es)

## Target
ALL languages F1 >= 0.75 for paper quality

## Approach
1. Load V8.1 dataset from Google Drive
2. Train XLM-R baseline (like StandUp4AI but with more data)
3. Evaluate per-language F1
4. Save best model


In [ ]:
!pip install -q torch transformers scikit-learn pandas numpy

In [ ]:
import torch
import pandas as pd
import numpy as np
import os
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report
from transformers import AutoModel, AutoTokenizer, get_linear_schedule_with_warmup
from torch.optim import AdamW
from torch.utils.data import Dataset, DataLoader
import warnings
warnings.filterwarnings('ignore')

# Config
MODEL_NAME = 'xlm-roberta-base'
MAX_LEN = 128
BATCH_SIZE = 16
EPOCHS = 10
LR = 2e-5
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# V8.1 data location
V8_DIR = '/content/drive/MyDrive/autonomous_laughter_prediction_essential/data/v8_1_final'
OUTPUT_DIR = '/content/drive/MyDrive/v8_1_models'
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f'Output: {OUTPUT_DIR}')

## Load V8.1 Dataset

In [ ]:
def load_jsonl(path):
    data = []
    with open(path, 'r') as f:
        for line in f:
            data.append(json.loads(line))
    return pd.DataFrame(data)

print('Loading V8.1 dataset...')
train_df = load_jsonl(f'{V8_DIR}/train.jsonl')
val_df = load_jsonl(f'{V8_DIR}/valid.jsonl')
test_df = load_jsonl(f'{V8_DIR}/test.jsonl')

print(f'Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}')
print(f'\nLanguages: {train_df["language"].value_counts().to_dict()}')

In [ ]:
import json

# Check for label column - V8.1 might have 'label' not 'labels'
print('Train columns:', train_df.columns.tolist())
print('Sample:', train_df.iloc[0].to_dict())

In [ ]:
# For V8.1, we need to handle both word-level and sentence-level
# Check if we have word-level labels
if 'labels' in train_df.columns:
    # Word-level - compute example-level label (any positive = positive)
    train_df['example_label'] = train_df['labels'].apply(lambda x: 1 if sum(x) > 0 else 0)
    val_df['example_label'] = val_df['labels'].apply(lambda x: 1 if sum(x) > 0 else 0)
    test_df['example_label'] = test_df['labels'].apply(lambda x: 1 if sum(x) > 0 else 0)
elif 'label' in train_df.columns:
    train_df['example_label'] = train_df['label']
    val_df['example_label'] = val_df['label']
    test_df['example_label'] = test_df['label']
else:
    print('Warning: No label column found')
    
print(f'Example-level label dist:\n{train_df["example_label"].value_counts()}')

## Dataset Class

In [ ]:
class V8Dataset(Dataset):
    def __init__(self, texts, labels, tok, max_len):
        self.texts = texts
        self.labels = labels
        self.tok = tok
        self.max_len = max_len
    
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        # Join words into text
        if isinstance(self.texts[idx], list):
            text = ' '.join(str(w) for w in self.texts[idx])
        else:
            text = str(self.texts[idx])
        
        label = self.labels[idx]
        
        enc = self.tok(text, add_special_tokens=True, max_length=self.max_len, 
                       padding='max_length', truncation=True, return_attention_mask=True, return_tensors='pt')
        
        return {
            'input_ids': enc['input_ids'].flatten(),
            'attention_mask': enc['attention_mask'].flatten(),
            'label': torch.tensor(label, dtype=torch.long)
        }

In [ ]:
class XLMRClassifier(torch.nn.Module):
    def __init__(self, model_name):
        super().__init__()
        self.xlmr = AutoModel.from_pretrained(model_name)
        self.dropout = torch.nn.Dropout(0.2)
        self.classifier = torch.nn.Linear(self.xlmr.config.hidden_size, 2)
    
    def forward(self, input_ids, attention_mask):
        outputs = self.xlmr(input_ids=input_ids, attention_mask=attention_mask)
        pooled = outputs.last_hidden_state[:, 0]
        pooled = self.dropout(pooled)
        return self.classifier(pooled)

## Prepare Data

In [ ]:
# Prepare train/val data
train_texts = train_df['words'].apply(lambda x: ' '.join(x)).tolist() if 'words' in train_df.columns else train_df['text'].tolist()
train_labels = train_df['example_label'].tolist()

val_texts = val_df['words'].apply(lambda x: ' '.join(x)).tolist() if 'words' in val_df.columns else val_df['text'].tolist()
val_labels = val_df['example_label'].tolist()

print(f'Train: {len(train_texts)}, Val: {len(val_texts)}')

# Sample train for faster iteration (use 4000 for quick training, full 12048 for final)
SAMPLE_SIZE = min(4000, len(train_texts))
indices = np.random.RandomState(42).permutation(len(train_texts))[:SAMPLE_SIZE]
train_texts_sample = [train_texts[i] for i in indices]
train_labels_sample = [train_labels[i] for i in indices]
print(f'Using {SAMPLE_SIZE} samples for training')

In [ ]:
# Tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Create datasets
train_ds = V8Dataset(train_texts_sample, train_labels_sample, tokenizer, MAX_LEN)
val_ds = V8Dataset(val_texts, val_labels, tokenizer, MAX_LEN)

train_ld = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_ld = DataLoader(val_ds, batch_size=BATCH_SIZE)

print(f'Train batches: {len(train_ld)}, Val batches: {len(val_ld)}')

In [ ]:
# Model
model = XLMRClassifier(MODEL_NAME).to(DEVICE)
optimizer = AdamW(model.parameters(), lr=LR, weight_decay=0.01)
total_steps = len(train_ld) * EPOCHS
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=int(0.1*total_steps), num_training_steps=total_steps)
print('Model ready!')

## Training Functions

In [ ]:
def train_epoch(model, loader, optimizer, scheduler, device):
    model.train()
    total_loss = 0
    preds, labels = [], []
    criterion = torch.nn.CrossEntropyLoss()
    
    for batch in loader:
        optimizer.zero_grad()
        logits = model(batch['input_ids'].to(device), batch['attention_mask'].to(device))
        loss = criterion(logits, batch['label'].to(device))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        
        total_loss += loss.item()
        preds.extend(torch.argmax(logits, 1).cpu().numpy())
        labels.extend(batch['label'].numpy())
    
    return total_loss / len(loader), f1_score(labels, preds, average='weighted')

def eval_epoch(model, loader, device):
    model.eval()
    total_loss = 0
    preds, labels = [], []
    criterion = torch.nn.CrossEntropyLoss()
    
    with torch.no_grad():
        for batch in loader:
            logits = model(batch['input_ids'].to(device), batch['attention_mask'].to(device))
            loss = criterion(logits, batch['label'].to(device))
            total_loss += loss.item()
            preds.extend(torch.argmax(logits, 1).cpu().numpy())
            labels.extend(batch['label'].numpy())
    
    return total_loss / len(loader), f1_score(labels, preds, average='weighted'), preds, labels

## Training Loop

In [ ]:
best_f1 = 0
history = {'train_loss': [], 'train_f1': [], 'val_loss': [], 'val_f1': []}

print('Training V8.1...')
for epoch in range(EPOCHS):
    tl, tf = train_epoch(model, train_ld, optimizer, scheduler, DEVICE)
    vl, vf, _, _ = eval_epoch(model, val_ld, DEVICE)
    
    history['train_loss'].append(tl)
    history['train_f1'].append(tf)
    history['val_loss'].append(vl)
    history['val_f1'].append(vf)
    
    print(f'Epoch {epoch+1}/{EPOCHS}: TL={tl:.4f} TF={tf:.4f} VL={vl:.4f} VF={vf:.4f}')
    
    if vf > best_f1:
        best_f1 = vf
        torch.save(model.state_dict(), f'{OUTPUT_DIR}/best_v8.pt')
        print(f'  -> Saved best model! F1={best_f1:.4f}')

print(f'\nBest Val F1: {best_f1:.4f}')

## Per-Language Evaluation

In [ ]:
model.load_state_dict(torch.load(f'{OUTPUT_DIR}/best_v8.pt'))
_, _, val_preds, val_labels = eval_epoch(model, val_ld, DEVICE)

# Add predictions to val_df
val_df_eval = val_df.copy()
val_df_eval['pred'] = val_preds

print('='*50)
print('PER-LANGUAGE RESULTS')
print('='*50)

results = {}
for lang in val_df_eval['language'].unique():
    ld = val_df_eval[val_df_eval['language'] == lang]
    if len(ld) > 0:
        f1 = f1_score(ld['example_label'], ld['pred'], average='weighted')
        results[lang] = f1
        status = 'PASS' if f1 >= 0.75 else 'FAIL'
        print(f'{lang.upper()}: F1={f1:.4f} [{status}] (n={len(ld)})')

overall = f1_score(val_labels, val_preds, average='weighted')
status = 'PASS' if overall >= 0.75 else 'FAIL'
print(f'\nOVERALL: F1={overall:.4f} [{status}]')

In [ ]:
import json

# Save results
with open(f'{OUTPUT_DIR}/v8_results.json', 'w') as f:
    json.dump({'overall': overall, 'per_language': results, 'best_f1': best_f1, 'history': history}, f, indent=2)

model.save_pretrained(f'{OUTPUT_DIR}/xlmr_v8')
tokenizer.save_pretrained(f'{OUTPUT_DIR}/xlmr_v8')

print('\nSaved to:', OUTPUT_DIR)
print('Done!')

## Summary

**Target:** All languages F1 >= 0.75

**Dataset:** V8.1 (12,048 examples)

**Results:** Per-language F1 scores above
